In [3]:
import mlflow  # Importuje MLflow do rejestrowania eksperymentów
import mlflow.sklearn  # Importuje integrację MLflow ze scikit-learn
import pandas as pd  # Importuje Pandas do pracy z danymi tabelarycznymi
from sklearn.model_selection import train_test_split, GridSearchCV  # Importuje podział danych i strojenie parametrów
from sklearn.linear_model import LogisticRegression  # Importuje model regresji logistycznej
from sklearn.metrics import accuracy_score  # Importuje funkcję obliczającą dokładność
from sklearn.preprocessing import StandardScaler  # Importuje narzędzie do standaryzacji danych

mlflow.set_tracking_uri("http://127.0.0.1:8001")  # Łączy notebook z lokalnym serwerem MLflow
mlflow.set_experiment("Analiza danych Titanic")  # Tworzy lub wybiera eksperyment Titanic
mlflow.sklearn.autolog()  # Włącza automatyczne rejestrowanie parametrów, metryk i modelu

# Ładowanie danych
titanic = pd.read_csv(  # Rozpoczyna pobieranie zbioru danych Titanic
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"  # Podaje adres pliku CSV
)  # Kończy pobieranie danych

# Przetwarzanie danych
titanic.drop(  # Rozpoczyna usuwanie niepotrzebnych kolumn
    ["Name", "Ticket", "Cabin"],  # Wskazuje kolumny przeznaczone do usunięcia
    axis=1,  # Informuje, że usuwane są kolumny
    inplace=True  # Wprowadza zmianę bezpośrednio w tabeli
)  # Kończy usuwanie kolumn

titanic["Age"] = titanic["Age"].fillna(  # Rozpoczyna uzupełnianie brakujących wartości wieku
    titanic["Age"].median()  # Uzupełnia braki medianą wieku
)  # Kończy uzupełnianie wieku

titanic["Embarked"] = titanic["Embarked"].fillna(  # Rozpoczyna uzupełnianie brakujących portów
    titanic["Embarked"].mode()[0]  # Uzupełnia braki najczęściej występującym portem
)  # Kończy uzupełnianie portów

titanic = pd.get_dummies(  # Rozpoczyna zamianę zmiennych tekstowych na liczbowe
    titanic,  # Przekazuje tabelę Titanic
    drop_first=True  # Pomija pierwszą kategorię każdej cechy
)  # Kończy zamianę danych tekstowych

X = titanic.drop(  # Rozpoczyna tworzenie zbioru cech
    "Survived",  # Wskazuje kolumnę, która nie będzie cechą
    axis=1  # Informuje, że usuwana jest kolumna
)  # Kończy tworzenie zbioru cech

y = titanic["Survived"]  # Tworzy zmienną docelową informującą o przeżyciu

# Podział na dane treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(  # Rozpoczyna podział danych
    X,  # Przekazuje cechy modelu
    y,  # Przekazuje zmienną docelową
    test_size=0.2,  # Przeznacza 20 procent danych na testowanie
    random_state=42,  # Zapewnia taki sam podział przy każdym uruchomieniu
    stratify=y  # Zachowuje proporcje osób, które przeżyły i nie przeżyły
)  # Kończy podział danych

training_data = titanic.loc[X_train.index].copy()  # Pobiera wiersze wykorzystane do trenowania modelu

# Skalowanie danych
scaler = StandardScaler()  # Tworzy obiekt standaryzujący cechy
X_train_scaled = scaler.fit_transform(X_train)  # Uczy skalowanie wyłącznie na danych treningowych
X_test_scaled = scaler.transform(X_test)  # Skaluje dane testowe według parametrów danych treningowych

# Definicja hiperparametrów do strojenia
param_grid = {  # Rozpoczyna słownik testowanych hiperparametrów
    "C": [0.1, 1, 10],  # Określa testowane wartości regularyzacji
    "solver": ["liblinear", "lbfgs"]  # Określa testowane algorytmy optymalizacyjne
}  # Kończy słownik hiperparametrów

# Inicjalizacja modelu
logreg = LogisticRegression(  # Rozpoczyna tworzenie modelu regresji logistycznej
    max_iter=200,  # Ustawia maksymalną liczbę iteracji uczenia
    random_state=42  # Zapewnia powtarzalność działania modelu
)  # Kończy tworzenie modelu

# Konfiguracja GridSearchCV
grid_search = GridSearchCV(  # Rozpoczyna konfigurację wyszukiwania parametrów
    estimator=logreg,  # Przekazuje model regresji logistycznej
    param_grid=param_grid,  # Przekazuje testowane hiperparametry
    cv=3,  # Ustawia trzykrotną walidację krzyżową
    n_jobs=-1,  # Pozwala użyć wszystkich dostępnych rdzeni procesora
    scoring="accuracy"  # Wybiera dokładność jako kryterium oceny
)  # Kończy konfigurację GridSearchCV

# Uruchomienie przebiegu MLflow
with mlflow.start_run(run_name="Titanic GridSearch") as run:  # Rozpoczyna przebieg MLflow

    training_data_for_mlflow = training_data.reset_index(drop=True)  # Porządkuje indeks datasetu

    mlflow_dataset = mlflow.data.from_pandas(  # Tworzy obiekt datasetu MLflow
        training_data_for_mlflow,  # Przekazuje dane treningowe
        source="https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",  # Zapisuje źródło
        targets="Survived",  # Wskazuje zmienną docelową
        name="Titanic Training Dataset"  # Nadaje datasetowi nazwę
    )  # Kończy tworzenie datasetu

    mlflow.log_input(  # Rejestruje metadane datasetu w bieżącym runie
        mlflow_dataset,  # Przekazuje dataset MLflow
        context="training"  # Oznacza go jako dane treningowe
    )  # Kończy rejestrowanie datasetu

    training_data_for_mlflow.to_csv(  # Rozpoczyna zapis pełnego datasetu do pliku
        "titanic_training.csv",  # Ustawia nazwę pliku
        index=False  # Nie zapisuje indeksu Pandas
    )  # Kończy zapis pliku CSV

    mlflow.log_artifact(  # Zapisuje pełny plik CSV w MLflow
        "titanic_training.csv",  # Wskazuje plik do zapisania
        artifact_path="datasets"  # Umieszcza plik w folderze datasets
    )  # Kończy zapisywanie pliku

    grid_search.fit(X_train_scaled, y_train)  # Trenuje model i wyszukuje parametry

    best_model = grid_search.best_estimator_  # Pobiera najlepszy model
    y_pred = best_model.predict(X_test_scaled)  # Wykonuje predykcje testowe
    accuracy = accuracy_score(y_test, y_pred)  # Oblicza dokładność testową

    mlflow.log_metric("test_accuracy", accuracy)  # Zapisuje dokładność testową
    mlflow.log_param("selected_C", grid_search.best_params_["C"])  # Zapisuje najlepsze C
    mlflow.log_param("selected_solver", grid_search.best_params_["solver"])  # Zapisuje solver
    mlflow.set_tag("model_description", "Regresja logistyczna dla danych Titanic")  # Dodaje opis

    print("ID przebiegu:", run.info.run_id)  # Wyświetla identyfikator runu
    print("Dokładność testowa:", accuracy)  # Wyświetla dokładność
    print("Najlepsze parametry:", grid_search.best_params_)  # Wyświetla parametry

C:\Users\micha\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/03 14:08:36 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/08/03 14:08:49 WARNING mlflow.sklearn: Saving scikit

ID przebiegu: 5ca4b76d23984ad7aa3fea6ce283d601
Dokładność testowa: 0.7988826815642458
Najlepsze parametry: {'C': 0.1, 'solver': 'lbfgs'}
🏃 View run Titanic GridSearch at: http://127.0.0.1:8001/#/experiments/1/runs/5ca4b76d23984ad7aa3fea6ce283d601
🧪 View experiment at: http://127.0.0.1:8001/#/experiments/1


 RADY OD CHATA
 
Przy każdym eksperymencie:
- Ustaw jeden wspólny eksperyment dla konkretnego problemu.
- Nadaj przebiegowi opisową nazwę.
- Zapisz wersję danych i sposób podziału.
- Zapisz parametry modelu.
- Trenuj model.
- Zapisz metryki walidacyjne i testowe.
- Zapisz model, sygnaturę oraz wykresy.
- Porównaj przebieg z wcześniejszym modelem bazowym.
- Wybierz model na podstawie kilku metryk i stabilności.
- Dodaj tag informujący, dlaczego model został wybrany.

Najważniejsza zasada: MLflow nie mówi automatycznie, który model jest najlepszy. Porządkuje dowody, dzięki którym możesz podjąć taką decyzję. Najlepszy model to zwykle nie ten z najwyższą pojedynczą liczbą, tylko ten, który osiąga dobre, stabilne wyniki na niewidzianych danych i można go odtworzyć.


- Jupyter/Colab/skrypt — laboratorium, w którym uruchamiasz modele,
- MLflow Tracking — dziennik zapisujący przebiegi,
- MLflow UI — panel do oglądania i porównywania,
- MLflow Model Registry — katalog wybranych modeli gotowych do dalszego wykorzystania.
- MLflow ma także funkcje uruchamiania projektów i serwowania modeli, ale przy typowej nauce i eksperymentowaniu model trenujesz w notebooku, a MLflow rejestruje rezultaty.

4. Porównywanie modeli

Spróbujmy teraz porównać wytrenowaną przez nas regresję logistyczną z bardziej zaawansowaną metodą uczenia maszynowego jaką jest las losowy (ang. RandomForest). Zalogujemy następnie następnie przebieg takiego treningu i za pomocą MLFlow będziemy w stanie porównać działaniu obu tych modeli. Spójrzmy na poniższy kod.

In [5]:
import mlflow  # Importuje MLflow do rejestrowania eksperymentów i modeli
import pandas as pd  # Importuje Pandas do pracy z danymi tabelarycznymi
from sklearn.model_selection import train_test_split, GridSearchCV  # Importuje podział danych i wyszukiwanie parametrów
from sklearn.ensemble import RandomForestClassifier  # Importuje klasyfikator Random Forest
from sklearn.metrics import accuracy_score  # Importuje funkcję obliczającą dokładność modelu

mlflow.set_tracking_uri("http://localhost:8001")  # Łączy kod z lokalnym serwerem MLflow
mlflow.set_experiment("Analiza danych Titanic - Random Forest")  # Tworzy lub wybiera eksperyment MLflow
mlflow.sklearn.autolog(  # Włącza automatyczne rejestrowanie modelu i wyników
    log_datasets=False,  # Wyłącza automatyczne logowanie datasetu, które może powodować ostrzeżenia
    max_tuning_runs=None  # Rejestruje wszystkie kombinacje parametrów GridSearchCV
)  # Kończy konfigurację autologowania

# Ładowanie danych
titanic = pd.read_csv(  # Rozpoczyna pobieranie zbioru danych Titanic
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"  # Podaje adres pliku CSV
)  # Kończy pobieranie danych

# Przetwarzanie danych
titanic.drop(  # Rozpoczyna usuwanie niepotrzebnych kolumn
    ["PassengerId", "Name", "Ticket", "Cabin"],  # Wskazuje kolumny przeznaczone do usunięcia
    axis=1,  # Informuje, że usuwane są kolumny
    inplace=True  # Wprowadza zmiany bezpośrednio w tabeli
)  # Kończy usuwanie kolumn

titanic["Age"] = titanic["Age"].fillna(  # Rozpoczyna uzupełnianie brakujących wartości wieku
    titanic["Age"].median()  # Uzupełnia braki medianą wieku
)  # Kończy uzupełnianie wieku

titanic["Embarked"] = titanic["Embarked"].fillna(  # Rozpoczyna uzupełnianie brakujących portów
    titanic["Embarked"].mode()[0]  # Uzupełnia braki najczęściej występującym portem
)  # Kończy uzupełnianie portów

titanic = pd.get_dummies(  # Rozpoczyna zamianę zmiennych tekstowych na liczbowe
    titanic,  # Przekazuje tabelę Titanic
    drop_first=True  # Pomija pierwszą kategorię każdej zmiennej
)  # Kończy zamianę danych tekstowych

int_columns = titanic.select_dtypes(  # Rozpoczyna wyszukiwanie kolumn całkowitoliczbowych
    include="int"  # Wybiera kolumny typu integer
).columns  # Pobiera nazwy znalezionych kolumn

titanic[int_columns] = titanic[int_columns].astype(float)  # Zamienia kolumny całkowite na liczby zmiennoprzecinkowe

X = titanic.drop(  # Rozpoczyna tworzenie zbioru cech
    "Survived",  # Wskazuje kolumnę, która nie będzie cechą
    axis=1  # Informuje, że usuwana jest kolumna
)  # Kończy tworzenie zbioru cech

y = titanic["Survived"].to_numpy()  # Tworzy tablicę zawierającą informację o przeżyciu

# Podział na zestawy treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(  # Rozpoczyna podział danych
    X,  # Przekazuje cechy modelu
    y,  # Przekazuje zmienną docelową
    test_size=0.2,  # Przeznacza 20 procent danych na testowanie
    random_state=42,  # Zapewnia powtarzalność podziału
    stratify=y  # Zachowuje proporcje klas w obu zbiorach
)  # Kończy podział danych

# Definicja modelu Random Forest
rf_model = RandomForestClassifier(  # Rozpoczyna tworzenie modelu Random Forest
    random_state=42  # Zapewnia powtarzalność wyników modelu
)  # Kończy tworzenie modelu

# Definicja siatki hiperparametrów
param_grid = {  # Rozpoczyna słownik testowanych hiperparametrów
    "n_estimators": [50, 100, 200],  # Określa liczbę drzew w lesie
    "max_depth": [5, 10, 20, 30],  # Określa maksymalną głębokość drzew
    "min_samples_split": [2, 5, 10],  # Określa minimalną liczbę próbek potrzebnych do podziału węzła
    "min_samples_leaf": [1, 2, 4]  # Określa minimalną liczbę próbek w liściu
}  # Kończy słownik hiperparametrów

# Konfiguracja GridSearchCV
grid_search = GridSearchCV(  # Rozpoczyna konfigurację wyszukiwania parametrów
    estimator=rf_model,  # Przekazuje model Random Forest
    param_grid=param_grid,  # Przekazuje siatkę testowanych parametrów
    cv=5,  # Ustawia pięciokrotną walidację krzyżową
    n_jobs=-1,  # Pozwala wykorzystać wszystkie rdzenie procesora
    verbose=2,  # Wyświetla informacje o postępie obliczeń
    scoring="accuracy"  # Wybiera dokładność jako kryterium oceny
)  # Kończy konfigurację GridSearchCV

# Uruchomienie treningu i rejestrowanie w MLflow
with mlflow.start_run(run_name="Random Forest GridSearch") as run:  # Rozpoczyna przebieg MLflow

    grid_search.fit(X_train, y_train)  # Trenuje modele dla wszystkich kombinacji parametrów

    best_model = grid_search.best_estimator_  # Pobiera model z najlepszymi parametrami
    y_pred = best_model.predict(X_test)  # Wykonuje przewidywania dla danych testowych
    accuracy = accuracy_score(y_test, y_pred)  # Oblicza dokładność na zbiorze testowym

    mlflow.log_metric("test_accuracy", accuracy)  # Zapisuje dokładność testową w MLflow
    mlflow.log_param("selected_n_estimators", grid_search.best_params_["n_estimators"])  # Zapisuje najlepszą liczbę drzew
    mlflow.log_param("selected_max_depth", grid_search.best_params_["max_depth"])  # Zapisuje najlepszą głębokość drzew
    mlflow.log_param("selected_min_samples_split", grid_search.best_params_["min_samples_split"])  # Zapisuje najlepszy podział
    mlflow.log_param("selected_min_samples_leaf", grid_search.best_params_["min_samples_leaf"])  # Zapisuje najlepszą wielkość liścia

    print("ID przebiegu:", run.info.run_id)  # Wyświetla identyfikator przebiegu MLflow
    print("Dokładność testowa:", accuracy)  # Wyświetla dokładność najlepszego modelu
    print("Najlepsze parametry:", grid_search.best_params_)  # Wyświetla najlepszą konfigurację

Fitting 5 folds for each of 108 candidates, totalling 540 fits


2026/08/03 14:45:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/03 14:45:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run righteous-shrew-398 at: http://localhost:8001/#/experiments/2/runs/67b50ee210534b4ab1159b015caf4e07
🧪 View experiment at: http://localhost:8001/#/experiments/2
🏃 View run righteous-turtle-277 at: http://localhost:8001/#/experiments/2/runs/15757ff9e73c494bba63c5084ffe308b
🧪 View experiment at: http://localhost:8001/#/experiments/2
🏃 View run capable-chimp-205 at: http://localhost:8001/#/experiments/2/runs/ec0ab626040641beb9c58ec38574f629
🧪 View experiment at: http://localhost:8001/#/experiments/2
🏃 View run agreeable-sow-568 at: http://localhost:8001/#/experiments/2/runs/6386e6d1916841fdba7b8eabdf8341b0
🧪 View experiment at: http://localhost:8001/#/experiments/2
🏃 View run likeable-wren-320 at: http://localhost:8001/#/experiments/2/runs/e6a4616ef0bf4fe2a76d4478976c7d8d
🧪 View experiment at: http://localhost:8001/#/experiments/2
🏃 View run big-tern-656 at: http://localhost:8001/#/experiments/2/runs/ec726eaf986c48ca9b22ec75bdc17c06
🧪 View experiment at: http://localhost:8001/#/

19. Cwizenie

In [2]:
import tensorboard
import tensorflow as tf
import mlflow

print("TensorBoard:", tensorboard.__version__)
print("TensorFlow:", tf.__version__)
print("MLflow:", mlflow.__version__)

TensorBoard: 2.21.0
TensorFlow: 2.21.0
MLflow: 3.15.0


In [3]:
import mlflow  # Importuje MLflow do rejestrowania eksperymentów
import mlflow.tensorflow  # Importuje integrację MLflow z TensorFlow i Keras
import tensorflow as tf  # Importuje bibliotekę TensorFlow
from tensorflow.keras.datasets import imdb  # Importuje gotowy zbiór recenzji filmowych IMDB
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Importuje funkcję wyrównującą długość tekstów
from tensorflow.keras.models import Sequential  # Importuje sekwencyjny model sieci neuronowej
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense  # Importuje warstwy sieci neuronowej
from tensorflow.keras.regularizers import l2  # Importuje regularyzację L2
from sklearn.metrics import classification_report  # Importuje raport metryk klasyfikacji

mlflow.set_tracking_uri("http://localhost:8001")  # Łączy kod z lokalnym serwerem MLflow
mlflow.set_experiment("Analiza sentymentu IMDB")  # Tworzy lub wybiera eksperyment MLflow

mlflow.tensorflow.autolog(  # Włącza automatyczne rejestrowanie treningu Keras
    log_models=True,  # Zapisuje wytrenowany model jako artefakt
    log_datasets=True,  # Próbuje zapisać informacje o danych treningowych
    log_input_examples=True,  # Zapisuje przykładowe dane wejściowe
    log_model_signatures=True,  # Zapisuje format wejścia i wyjścia modelu
    log_every_epoch=True,  # Zapisuje metryki po każdej epoce
    checkpoint=True,  # Włącza zapisywanie punktu kontrolnego modelu
    checkpoint_monitor="val_loss",  # Obserwuje stratę walidacyjną
    checkpoint_mode="min",  # Uznaje mniejszą stratę walidacyjną za lepszą
    checkpoint_save_best_only=True  # Zachowuje najlepszy punkt kontrolny
)  # Kończy konfigurację autologowania

# Definicja hiperparametrów
num_words = 5000  # Ustawia liczbę najczęściej występujących słów w słowniku
maxlen = 200  # Ustawia jednakową maksymalną długość każdej recenzji
embedding_dim = 16  # Ustawia liczbę wymiarów wektora reprezentującego każde słowo
epochs = 10  # Ustawia maksymalną liczbę epok treningu
batch_size = 128  # Ustawia liczbę przykładów przetwarzanych jednocześnie
l2_strength = 0.01  # Ustawia siłę regularyzacji L2

# Pobieranie danych IMDB
(x_train, y_train), (x_test, y_test) = imdb.load_data(  # Pobiera treningowy i testowy zbiór recenzji
    num_words=num_words  # Pozostawia tylko najczęściej występujące słowa
)  # Kończy pobieranie danych

# Wyrównanie długości recenzji
x_train = pad_sequences(  # Rozpoczyna wyrównywanie danych treningowych
    x_train,  # Przekazuje recenzje treningowe
    maxlen=maxlen  # Ustawia długość każdej recenzji na 200 elementów
)  # Kończy wyrównywanie danych treningowych

x_test = pad_sequences(  # Rozpoczyna wyrównywanie danych testowych
    x_test,  # Przekazuje recenzje testowe
    maxlen=maxlen  # Ustawia długość każdej recenzji na 200 elementów
)  # Kończy wyrównywanie danych testowych

# Definicja funkcji budującej model
def build_keras_model(input_dim, output_dim, sequence_length):  # Definiuje funkcję tworzącą model Keras
    model = Sequential(  # Rozpoczyna tworzenie modelu sekwencyjnego
        [  # Rozpoczyna listę warstw modelu
            Input(shape=(sequence_length,)),  # Określa kształt pojedynczej recenzji
            Embedding(  # Rozpoczyna warstwę zamieniającą numery słów na wektory
                input_dim=input_dim,  # Ustawia wielkość słownika
                output_dim=output_dim,  # Ustawia liczbę wymiarów reprezentacji słowa
                embeddings_regularizer=l2(l2_strength)  # Dodaje regularyzację ograniczającą przeuczenie
            ),  # Kończy warstwę Embedding
            Flatten(),  # Zamienia macierz wektorów słów na jeden długi wektor
            Dense(  # Rozpoczyna tworzenie warstwy wyjściowej
                1,  # Ustawia jeden neuron wyjściowy
                activation="sigmoid"  # Zwraca prawdopodobieństwo pozytywnej recenzji
            )  # Kończy warstwę wyjściową
        ]  # Kończy listę warstw
    )  # Kończy tworzenie modelu

    model.compile(  # Rozpoczyna konfigurację procesu uczenia
        optimizer="adam",  # Ustawia optymalizator Adam
        loss="binary_crossentropy",  # Ustawia funkcję straty dla klasyfikacji binarnej
        metrics=["accuracy"]  # Ustawia dokładność jako metrykę treningową
    )  # Kończy konfigurację modelu

    return model  # Zwraca gotowy model Keras

# Rozpoczęcie przebiegu MLflow
with mlflow.start_run(run_name="Keras Embedding Sentiment") as run:  # Rozpoczyna nowy przebieg MLflow

    mlflow.log_param("num_words", num_words)  # Zapisuje wielkość słownika
    mlflow.log_param("maxlen", maxlen)  # Zapisuje maksymalną długość recenzji
    mlflow.log_param("embedding_dim", embedding_dim)  # Zapisuje wielkość embeddingu
    mlflow.log_param("l2_strength", l2_strength)  # Zapisuje siłę regularyzacji

    model = build_keras_model(  # Rozpoczyna budowanie sieci neuronowej
        input_dim=num_words,  # Przekazuje wielkość słownika
        output_dim=embedding_dim,  # Przekazuje wielkość wektora embeddingu
        sequence_length=maxlen  # Przekazuje długość recenzji
    )  # Kończy budowanie modelu

    history = model.fit(  # Rozpoczyna trening modelu
        x_train,  # Przekazuje treningowe recenzje
        y_train,  # Przekazuje treningowe etykiety sentymentu
        epochs=epochs,  # Ustawia liczbę epok
        batch_size=batch_size,  # Ustawia wielkość paczki
        validation_split=0.2,  # Przeznacza 20 procent danych treningowych na walidację
        verbose=1  # Wyświetla postęp treningu
    )  # Kończy trening modelu

    test_loss, test_accuracy = model.evaluate(  # Rozpoczyna ocenę modelu na danych testowych
        x_test,  # Przekazuje testowe recenzje
        y_test,  # Przekazuje prawdziwe etykiety testowe
        verbose=1  # Wyświetla postęp oceny
    )  # Kończy ocenę modelu

    probabilities = model.predict(  # Rozpoczyna przewidywanie prawdopodobieństw
        x_test,  # Przekazuje testowe recenzje
        verbose=1  # Wyświetla postęp przewidywania
    )  # Kończy przewidywanie prawdopodobieństw

    y_pred = (probabilities.flatten() > 0.5).astype("int32")  # Zamienia prawdopodobieństwa na klasy 0 lub 1

    report_text = classification_report(  # Rozpoczyna tworzenie tekstowego raportu klasyfikacji
        y_test,  # Przekazuje prawdziwe klasy
        y_pred,  # Przekazuje przewidziane klasy
        target_names=["Negative", "Positive"],  # Ustawia czytelne nazwy klas
        zero_division=0  # Zapobiega błędowi dzielenia przez zero
    )  # Kończy tworzenie raportu tekstowego

    report_dict = classification_report(  # Rozpoczyna tworzenie raportu w postaci słownika
        y_test,  # Przekazuje prawdziwe klasy
        y_pred,  # Przekazuje przewidziane klasy
        target_names=["Negative", "Positive"],  # Ustawia nazwy klas
        output_dict=True,  # Zwraca wyniki jako słownik
        zero_division=0  # Zapobiega błędowi dzielenia przez zero
    )  # Kończy tworzenie raportu jako słownika

    mlflow.log_metric("test_loss", test_loss)  # Ręcznie zapisuje stratę testową
    mlflow.log_metric("test_accuracy", test_accuracy)  # Ręcznie zapisuje dokładność testową
    mlflow.log_metric("test_precision", report_dict["macro avg"]["precision"])  # Zapisuje średnią precyzję
    mlflow.log_metric("test_recall", report_dict["macro avg"]["recall"])  # Zapisuje średnią czułość
    mlflow.log_metric("test_f1", report_dict["macro avg"]["f1-score"])  # Zapisuje średni wynik F1

    mlflow.log_text(  # Rozpoczyna zapisywanie raportu jako artefaktu
        report_text,  # Przekazuje tekst raportu klasyfikacji
        "reports/classification_report.txt"  # Ustawia lokalizację raportu w MLflow
    )  # Kończy zapisywanie raportu

    mlflow.log_dict(  # Rozpoczyna zapisywanie raportu jako pliku JSON
        report_dict,  # Przekazuje raport w postaci słownika
        "reports/classification_report.json"  # Ustawia lokalizację pliku JSON
    )  # Kończy zapisywanie raportu

    mlflow.set_tags(  # Rozpoczyna dodawanie opisów przebiegu
        {  # Rozpoczyna słownik znaczników
            "task": "sentiment_analysis",  # Określa rodzaj zadania
            "dataset": "IMDB",  # Określa użyty zbiór danych
            "framework": "TensorFlow Keras"  # Określa użyty framework
        }  # Kończy słownik znaczników
    )  # Kończy dodawanie znaczników

    print("ID przebiegu:", run.info.run_id)  # Wyświetla identyfikator przebiegu
    print("Test loss:", test_loss)  # Wyświetla stratę testową
    print("Test accuracy:", test_accuracy)  # Wyświetla dokładność testową
    print(report_text)  # Wyświetla raport klasyfikacji

Epoch 1/10
154/157 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5626 - loss: 0.7506

157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.5645 - loss: 0.7493 - val_accuracy: 0.6784 - val_loss: 0.6612
Epoch 2/10
154/157 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7461 - loss: 0.6089

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7469 - loss: 0.6086 - val_accuracy: 0.7832 - val_loss: 0.5692
Epoch 3/10
151/157 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8043 - loss: 0.5434

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8045 - loss: 0.5430 - val_accuracy: 0.8076 - val_loss: 0.5259
Epoch 4/10
155/157 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8250 - loss: 0.5081

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8246 - loss: 0.5083 - val_accuracy: 0.8166 - val_loss: 0.5066
Epoch 5/10
155/157 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8345 - loss: 0.4870

...

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8579 - loss: 0.4365 - val_accuracy: 0.8470 - val_loss: 0.4463
Epoch 10/10
151/157 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8609 - loss: 0.4276

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8604 - loss: 0.4280 - val_accuracy: 0.8492 - val_loss: 0.4429
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


2026/08/03 15:22:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/08/03 15:23:07 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\micha\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 5 variables whereas the saved optimizer has 8 variables. "


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8494 - loss: 0.4402
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
ID przebiegu: f0303402c24845de81b312b0bbcaac34
Test loss: 0.44021621346473694
Test accuracy: 0.8493599891662598
              precision    recall  f1-score   support

    Negative       0.82      0.89      0.85     12500
    Positive       0.88      0.81      0.84     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.85     25000
weighted avg       0.85      0.85      0.85     25000

🏃 View run Keras Embedding Sentiment at: http://localhost:8001/#/experiments/1/runs/f0303402c24845de81b312b0bbcaac34
🧪 View experiment at: http://localhost:8001/#/experiments/1


Wszystko zadziałało prawidłowo — model osiągnął około 84,94% dokładności na 25 000 recenzji testowych, a cały przebieg został zapisany w MLflow.
Interpretacja wyników:
Test loss: 0.4402 — wartość funkcji straty; przy porównywaniu podobnych modeli niższa wartość jest zwykle lepsza.
Test accuracy: 0.8494 — około 85 na 100 recenzji zostało sklasyfikowanych poprawnie.
Negative recall: 0.89 — model rozpoznał 89% negatywnych recenzji.
Positive recall: 0.81 — rozpoznał 81% pozytywnych recenzji.
Positive precision: 0.88 — jeśli model oznaczył recenzję jako pozytywną, miał rację w 88% przypadków.
macro F1: 0.85 — model osiąga ogólnie dobry i dość zrównoważony wynik dla obu klas.
Model trochę lepiej wykrywa recenzje negatywne niż pozytywne:
Recall negatywnych: 0.89
Recall pozytywnych: 0.81
W MLflow możesz teraz:
Otworzyć bezpośrednio przebieg Keras Embedding Sentiment.
W Metrics obejrzeć wykresy loss, accuracy, val_loss i val_accuracy dla kolejnych epok.
Porównać końcowe test_accuracy, test_precision, test_recall i test_f1.
W Artifacts sprawdzić model, checkpoint, logi TensorBoard i raport klasyfikacji.
Uruchomić kolejne próby, zmieniając np. embedding_dim, l2_strength, liczbę epok albo architekturę.
Ten przebieg jest dobrym modelem bazowym. Kolejne modele warto porównywać właśnie z jego test_accuracy ≈ 0.849 i test_f1 ≈ 0.85.